# Gravitational Clustering Algorithm (GCA)
## Entendimiento del algoritmo base antes de sus variaciones y evoluciones

**Objetivo:** Implementar el algoritmo de clustering gravitacional en su versión clásica, basado en la ley de gravitación universal de Newton.

**Hipótesis:** Los algoritmos de clustering gravitacional, en esencia, siguen usando conceptos de la física newtoniana — cada punto de datos se trata como una partícula con masa, y la fuerza gravitacional entre partículas las atrae mutuamente hasta formar agrupaciones naturales.

### Formulación matemática

La fuerza gravitacional entre dos partículas $i$ y $j$:

$$F_{ij} = G \cdot \frac{m_i \cdot m_j}{\|x_i - x_j\|^2 + \varepsilon}$$

Donde:
- $G$ = constante gravitacional (controla la intensidad de atracción)
- $m_i, m_j$ = masas de las partículas (inicialmente proporcionales a la densidad local)
- $\|x_i - x_j\|$ = distancia euclidiana entre partículas
- $\varepsilon$ = softening parameter (evita singularidades cuando $r \to 0$)

**Actualización de posiciones** (en cada iteración $t$):

$$v_i^{(t+1)} = \alpha \cdot v_i^{(t)} + \frac{\sum_{j \neq i} F_{ij} \cdot \hat{r}_{ij}}{m_i}$$

$$x_i^{(t+1)} = x_i^{(t)} + v_i^{(t+1)}$$

Donde $\hat{r}_{ij}$ es el vector unitario de $i$ hacia $j$ y $\alpha$ es un factor de amortiguamiento.

### Parámetros del experimento
| Parámetro | Rango | Descripción |
|-----------|-------|-------------|
| G | 0.1 — 1.0 | Constante gravitacional |
| ε (epsilon) | 1e-3 — 1e-1 | Softening (evita div/0) |
| α (damping) | 0.0 — 0.9 | Amortiguamiento de velocidad |
| T (temperatura) | 0.0 — 0.5 | Ruido térmico (repulsión) |
| Iteraciones | 50 — 200 | Máximo de pasos |

In [ ]:
# --- Celda 1: Imports y configuración ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from scipy.spatial.distance import cdist
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    calinski_harabasz_score,
)
from IPython.display import HTML
import time
import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo de gráficos
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})



## 1. Generación de datos sintéticos

Generamos datos con `make_blobs` para tener clusters bien definidos con ground truth.  
Se añade ruido gaussiano (σ=0.1) y un porcentaje configurable de outliers uniformes.

In [ ]:
# --- Celda 2: Generación de datos sintéticos ---

def generate_data(n_points=300, n_clusters=4, outlier_frac=0.05, noise_std=0.1, seed=SEED):
    """
    Genera datos sintéticos con clusters, ruido gaussiano y outliers.
    
    Returns: X (datos), y_true (etiquetas, -1 para outliers)
    """
    rng = np.random.RandomState(seed)
    
    # Puntos base con clusters
    n_clean = int(n_points * (1 - outlier_frac))
    n_outliers = n_points - n_clean
    
    X_clean, y_clean = make_blobs(
        n_samples=n_clean,
        n_features=2,
        centers=n_clusters,
        cluster_std=0.8,
        random_state=seed,
    )
    
    # Ruido gaussiano adicional
    X_clean += rng.normal(0, noise_std, X_clean.shape)
    
    # Outliers uniformes en el rango extendido
    if n_outliers > 0:
        margin = 2.0
        x_min, x_max = X_clean[:, 0].min() - margin, X_clean[:, 0].max() + margin
        y_min, y_max = X_clean[:, 1].min() - margin, X_clean[:, 1].max() + margin
        X_outliers = rng.uniform(
            [x_min, y_min], [x_max, y_max], size=(n_outliers, 2)
        )
        y_outliers = np.full(n_outliers, -1)
        X = np.vstack([X_clean, X_outliers])
        y = np.concatenate([y_clean, y_outliers])
    else:
        X, y = X_clean, y_clean
    
    # Shuffle
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


# === PARÁMETROS CONFIGURABLES ===
N_POINTS = 300
N_CLUSTERS = 4
OUTLIER_FRAC = 0.05    # Cambiar a 0.0, 0.10, 0.20 para experimentos
NOISE_STD = 0.1

X, y_true = generate_data(N_POINTS, N_CLUSTERS, OUTLIER_FRAC, NOISE_STD)

print(f"Datos generados: {X.shape[0]} puntos, {N_CLUSTERS} clusters, "
      f"{(y_true == -1).sum()} outliers ({OUTLIER_FRAC*100:.0f}%)")
print(f"Rango X: [{X[:,0].min():.2f}, {X[:,0].max():.2f}]")
print(f"Rango Y: [{X[:,1].min():.2f}, {X[:,1].max():.2f}]")

## 2. Análisis exploratorio (EDA)

Visualización de los datos originales con su ground truth y estadísticas básicas por cluster.

In [ ]:
# --- Celda 3: EDA y visualización de datos originales ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a. Ground truth
colors = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS + 1))
for label in sorted(set(y_true)):
    mask = y_true == label
    lbl = f"Cluster {label}" if label >= 0 else "Outlier"
    marker = 'o' if label >= 0 else 'x'
    alpha = 0.7 if label >= 0 else 1.0
    axes[0].scatter(X[mask, 0], X[mask, 1], label=lbl, marker=marker,
                    alpha=alpha, s=30, edgecolors='k', linewidths=0.3)
axes[0].set_title("Ground Truth")
axes[0].legend(fontsize=9)

# 3b. Distribución de distancias entre puntos
from scipy.spatial.distance import pdist
dists = pdist(X)
axes[1].hist(dists, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(np.median(dists), color='red', ls='--', label=f'Mediana={np.median(dists):.2f}')
axes[1].set_title("Distribución de distancias par-a-par")
axes[1].set_xlabel("Distancia euclidiana")
axes[1].legend()

# 3c. Densidad local (k-vecinos más cercanos)
from sklearn.neighbors import NearestNeighbors
k = 10
nn = NearestNeighbors(n_neighbors=k+1).fit(X)
distances_knn, _ = nn.kneighbors(X)
local_density = 1.0 / (distances_knn[:, 1:].mean(axis=1) + 1e-10)

sc = axes[2].scatter(X[:, 0], X[:, 1], c=local_density, cmap='hot', s=30,
                     edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ax=axes[2], label="Densidad local (1/dist_media_kNN)")
axes[2].set_title(f"Densidad local (k={k} vecinos)")

plt.suptitle("Análisis Exploratorio de Datos", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Estadísticas por cluster
print("\n--- Estadísticas por cluster ---")
for label in sorted(set(y_true)):
    mask = y_true == label
    name = f"Cluster {label}" if label >= 0 else "Outliers"
    print(f"{name:>12}: n={mask.sum():3d}, "
          f"centroide=({X[mask,0].mean():+.2f}, {X[mask,1].mean():+.2f}), "
          f"std=({X[mask,0].std():.2f}, {X[mask,1].std():.2f})")

## 3. Preprocesamiento

Estandarización para que las distancias gravitacionales sean comparables entre dimensiones.  
Inicialización de masas proporcionales a la densidad local (puntos en zonas densas atraen más).

In [ ]:
# --- Celda 4: Preprocesamiento ---

# Estandarizar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Inicializar masas basadas en densidad local
# Puntos en zonas densas → mayor masa → atraen más
nn = NearestNeighbors(n_neighbors=11).fit(X_scaled)
dist_knn, _ = nn.kneighbors(X_scaled)
density = 1.0 / (dist_knn[:, 1:].mean(axis=1) + 1e-10)

# Normalizar masas al rango [0.1, 1.0]
masses = (density - density.min()) / (density.max() - density.min()) * 0.9 + 0.1

print(f"Datos estandarizados: media={X_scaled.mean(axis=0).round(4)}, std={X_scaled.std(axis=0).round(4)}")
print(f"Masas: min={masses.min():.3f}, max={masses.max():.3f}, media={masses.mean():.3f}")

# Visualizar masas
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=masses, cmap='viridis',
                s=masses * 80, edgecolors='k', linewidths=0.3, alpha=0.8)
plt.colorbar(sc, label="Masa (proporcional a densidad local)")
ax.set_title("Datos estandarizados con masas iniciales")
ax.set_xlabel("Feature 0 (scaled)")
ax.set_ylabel("Feature 1 (scaled)")
plt.tight_layout()
plt.show()

## 4. Implementación del Gravitational Clustering Algorithm (GCA)

### Algoritmo paso a paso:
1. **Inicializar** posiciones = datos, velocidades = 0, masas = densidad local
2. **Para cada iteración:**
   - Calcular fuerzas gravitacionales entre todos los pares de partículas activas
   - Opcionalmente añadir ruido térmico (temperatura) como perturbación
   - Actualizar velocidades con amortiguamiento (damping)
   - Actualizar posiciones
   - **Fusionar partículas** cuya distancia < `merge_dist` → la partícula resultante hereda la posición del centro de masa y la suma de masas
   - Registrar desplazamiento total para medir convergencia
3. **Criterio de parada:** desplazamiento total < umbral ó máximo de iteraciones
4. **Asignar clusters:** cada partícula superviviente ES un cluster; los puntos originales heredan la etiqueta de la partícula que los absorbió

> **Sin DBSCAN ni algoritmos externos.** Los clusters emergen puramente de la dinámica gravitacional + fusión.

In [ ]:
# --- Celda 5: Implementación del GCA (puro, sin DBSCAN) ---

class GravitationalClustering:
    """
    Gravitational Clustering Algorithm (GCA) puro.
    
    Cada punto es una partícula con masa. Las partículas se atraen
    según la ley de gravitación universal. Cuando dos partículas se
    acercan lo suficiente, se FUSIONAN en una sola (centro de masa + 
    masa combinada). Los clusters emergen de esta dinámica.
    """
    
    def __init__(self, G=0.5, epsilon=1e-2, damping=0.5, temperature=0.0,
                 max_iter=100, tol=1e-4, merge_dist=0.5, seed=42):
        self.G = G                    # Constante gravitacional
        self.epsilon = epsilon        # Softening parameter
        self.damping = damping        # Amortiguamiento de velocidad [0,1)
        self.temperature = temperature  # Ruido térmico
        self.max_iter = max_iter
        self.tol = tol                # Tolerancia de convergencia
        self.merge_dist = merge_dist  # Distancia para fusionar partículas
        self.seed = seed
    
    def fit(self, X, masses=None):
        """
        Ejecuta el algoritmo GCA con fusión de partículas.
        
        Parameters:
            X: array (n_samples, n_features) - datos de entrada
            masses: array (n_samples,) - masas iniciales (None = uniformes)
        
        Returns: self
        """
        rng = np.random.RandomState(self.seed)
        n, d = X.shape
        
        # --- Inicialización ---
        positions = X.copy()
        velocities = np.zeros_like(X)
        if masses is None:
            masses = np.ones(n)
        else:
            masses = masses.copy()
        
        # Cada punto empieza como su propia "partícula"
        # membership[i] = lista de índices originales que esta partícula representa
        membership = [[i] for i in range(n)]
        active = np.ones(n, dtype=bool)  # partículas activas (no fusionadas)
        
        # Historial para análisis
        self.history_ = {
            'positions': [positions.copy()],
            'displacements': [],
            'n_particles': [n],
            'max_velocity': [],
            'merges_per_iter': [],
        }
        
        import time
        t_start = time.time()
        
        for iteration in range(self.max_iter):
            active_idx = np.where(active)[0]
            n_active = len(active_idx)
            
            if n_active <= 1:
                break
            
            # --- 1. Posiciones y masas activas ---
            pos_act = positions[active_idx]
            mass_act = masses[active_idx]
            vel_act = velocities[active_idx]
            
            # --- 2. Calcular fuerzas gravitacionales ---
            # diff[i,j] = pos[j] - pos[i]  (vector de i hacia j)
            diff = pos_act[np.newaxis, :, :] - pos_act[:, np.newaxis, :]  # (n_a, n_a, d)
            dist_sq = np.sum(diff ** 2, axis=2) + self.epsilon               # (n_a, n_a)
            
            # F_ij = G * m_i * m_j / (r^2 + eps)
            mass_product = mass_act[:, np.newaxis] * mass_act[np.newaxis, :]
            force_magnitude = self.G * mass_product / dist_sq
            
            # Vector unitario de i → j
            dist = np.sqrt(dist_sq)
            direction = diff / dist[:, :, np.newaxis]
            
            # Fuerza vectorial
            force_vectors = force_magnitude[:, :, np.newaxis] * direction
            # Quitar auto-interacción
            for dd in range(d):
                np.fill_diagonal(force_vectors[:, :, dd], 0)
            
            # Aceleración = F_total / m_i
            accel = force_vectors.sum(axis=1) / mass_act[:, np.newaxis]
            
            # --- 3. Ruido térmico ---
            if self.temperature > 0:
                accel += rng.normal(0, self.temperature, accel.shape)
            
            # --- 4. Actualizar velocidades y posiciones ---
            vel_act = self.damping * vel_act + accel
            old_pos = pos_act.copy()
            pos_act = pos_act + vel_act
            
            # Guardar de vuelta
            positions[active_idx] = pos_act
            velocities[active_idx] = vel_act
            
            # --- 5. FUSIONAR partículas cercanas ---
            # Recalcular distancias después del movimiento
            dists_after = cdist(pos_act, pos_act)
            np.fill_diagonal(dists_after, np.inf)
            
            merges_this_iter = 0
            merged_this_step = set()
            
            for i_local in range(n_active):
                if i_local in merged_this_step:
                    continue
                for j_local in range(i_local + 1, n_active):
                    if j_local in merged_this_step:
                        continue
                    if dists_after[i_local, j_local] < self.merge_dist:
                        # Fusionar j en i (centro de masa ponderado)
                        i_global = active_idx[i_local]
                        j_global = active_idx[j_local]
                        
                        mi, mj = masses[i_global], masses[j_global]
                        total_mass = mi + mj
                        
                        # Nueva posición = centro de masa
                        positions[i_global] = (mi * positions[i_global] + mj * positions[j_global]) / total_mass
                        # Nueva velocidad = promedio ponderado
                        velocities[i_global] = (mi * velocities[i_global] + mj * velocities[j_global]) / total_mass
                        # Nueva masa = suma
                        masses[i_global] = total_mass
                        
                        # Transferir membresía
                        membership[i_global].extend(membership[j_global])
                        membership[j_global] = []
                        
                        # Desactivar j
                        active[j_global] = False
                        merged_this_step.add(j_local)
                        merges_this_iter += 1
            
            # --- 6. Medir convergencia ---
            displacement = np.sqrt(np.sum((pos_act - old_pos) ** 2, axis=1)).mean()
            
            self.history_['positions'].append(positions.copy())
            self.history_['displacements'].append(displacement)
            self.history_['n_particles'].append(int(active.sum()))
            self.history_['max_velocity'].append(float(np.max(np.linalg.norm(vel_act, axis=1))))
            self.history_['merges_per_iter'].append(merges_this_iter)
            
            if displacement < self.tol and merges_this_iter == 0:
                break
        
        self.n_iter_ = iteration + 1
        self.elapsed_time_ = time.time() - t_start
        self.positions_final_ = positions
        self.masses_final_ = masses
        self.active_ = active
        self.membership_ = membership
        
        # --- 7. Asignar etiquetas ---
        # Cada partícula activa es un cluster
        self.labels_ = np.full(n, -1)
        cluster_id = 0
        self.cluster_info_ = []
        for i in range(n):
            if active[i] and len(membership[i]) > 0:
                for orig_idx in membership[i]:
                    self.labels_[orig_idx] = cluster_id
                self.cluster_info_.append({
                    'id': cluster_id,
                    'center': positions[i].copy(),
                    'mass': masses[i],
                    'size': len(membership[i]),
                })
                cluster_id += 1
        
        self.n_clusters_ = cluster_id
        return self
    
    def get_params_str(self):
        return (f"G={self.G}, ε={self.epsilon}, damping={self.damping}, "
                f"T={self.temperature}, merge_dist={self.merge_dist}")


# --- Ejecutar GCA ---
# GUÍA DE AFINACIÓN:
#   G ↑  → atracción más fuerte → menos clusters (colapso más rápido)
#   damping ↑ → más inercia → partículas acumulan momento → convergen más rápido
#   merge_dist ↑ → fusión más agresiva → menos clusters
#   En datos estandarizados (std≈1), merge_dist ∈ [0.5, 1.5] suele funcionar bien

gca = GravitationalClustering(
    G=0.5,              # Atracción moderada-fuerte
    epsilon=1e-2,       # Softening (evita singularidades)
    damping=0.5,        # Inercia: acumula momento para que partículas converjan
    temperature=0.0,    # Sin ruido térmico (determinista)
    max_iter=200,       # Suficientes iteraciones para convergencia
    tol=1e-5,           # Umbral de parada por desplazamiento
    merge_dist=1.0,     # Radio de fusión ≈ desv. std. intra-cluster
)

gca.fit(X_scaled, masses)

print(f"Convergencia: {gca.n_iter_} iteraciones en {gca.elapsed_time_:.3f}s")
print(f"Partículas activas finales: {gca.active_.sum()} (de {len(X_scaled)} originales)")
print(f"Clusters encontrados: {gca.n_clusters_}")
print(f"Desplazamiento final: {gca.history_['displacements'][-1]:.6f}")
print(f"\n--- Detalle de clusters ---")
for info in sorted(gca.cluster_info_, key=lambda x: x['size'], reverse=True):
    print(f"  Cluster {info['id']}: {info['size']:3d} puntos, "
          f"masa={info['mass']:.2f}, centro=({info['center'][0]:+.2f}, {info['center'][1]:+.2f})")


## 5. Visualización del proceso gravitacional

Veamos cómo las partículas migran hacia los centros de masa, la curva de convergencia, y el resultado final del clustering.

In [ ]:
# --- Celda 6: Visualización del proceso y convergencia ---

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Snapshots del movimiento gravitacional (coloreados por ground truth)
n_hist = len(gca.history_['positions'])
steps_to_show = [0, n_hist // 4, n_hist // 2, 3 * n_hist // 4, n_hist - 1]
steps_to_show = sorted(set(min(s, n_hist - 1) for s in steps_to_show))[:4]

for idx, step in enumerate(steps_to_show):
    ax = axes[idx // 2, idx % 2]
    pos = gca.history_['positions'][step]
    # Solo mostrar partículas activas en ese momento sería complejo,
    # mostramos todas las posiciones (las fusionadas quedan quietas)
    ax.scatter(pos[:, 0], pos[:, 1], c=y_true, cmap='tab10', s=15, alpha=0.6,
               edgecolors='k', linewidths=0.2)
    n_part = gca.history_['n_particles'][step]
    ax.set_title(f"Iteración {step} ({n_part} partículas)", fontweight='bold')
    ax.set_xlim(X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1)
    ax.set_ylim(X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1)

# Curva de convergencia
ax_conv = axes[1, 0]
ax_conv.semilogy(gca.history_['displacements'], 'b-', linewidth=2, label='Desplazamiento medio')
ax_conv.axhline(gca.tol, color='r', ls='--', alpha=0.7, label=f'Tolerancia={gca.tol}')
ax_conv.set_xlabel("Iteración")
ax_conv.set_ylabel("Desplazamiento medio (log)")
ax_conv.set_title("Convergencia", fontweight='bold')
ax_conv.legend()

# Número de partículas activas y fusiones por iteración
ax_merge = axes[1, 1]
ax_merge.plot(gca.history_['n_particles'], 'g-', linewidth=2, label='Partículas activas')
ax_merge.axhline(N_CLUSTERS, color='orange', ls=':', label=f'Ground truth = {N_CLUSTERS}')
ax_merge.set_xlabel("Iteración")
ax_merge.set_ylabel("Nº partículas")
ax_merge.set_title("Fusiones gravitacionales", fontweight='bold')
ax_merge.legend()

ax_merge2 = ax_merge.twinx()
ax_merge2.bar(range(len(gca.history_['merges_per_iter'])), 
              gca.history_['merges_per_iter'], alpha=0.3, color='red', label='Fusiones/iter')
ax_merge2.set_ylabel("Fusiones por iteración", color='red')

# Resultado final con labels del GCA
ax_result = axes[1, 2]
for label in sorted(set(gca.labels_)):
    mask = gca.labels_ == label
    ax_result.scatter(X_scaled[mask, 0], X_scaled[mask, 1], label=f"C{label}",
                      s=30, alpha=0.7, edgecolors='k', linewidths=0.3)
# Marcar centros de clusters
for info in gca.cluster_info_:
    ax_result.scatter(*info['center'], marker='*', s=200, c='red', 
                      edgecolors='k', linewidths=1, zorder=5)
ax_result.set_title(f"Resultado GCA ({gca.n_clusters_} clusters)", fontweight='bold')
ax_result.legend(fontsize=7, loc='best', ncol=2)

# Posiciones finales colapsadas
ax_final = axes[0, 2]
# Mostrar solo partículas activas con tamaño proporcional a masa
active_idx = np.where(gca.active_)[0]
for i in active_idx:
    size = len(gca.membership_[i])
    ax_final.scatter(gca.positions_final_[i, 0], gca.positions_final_[i, 1],
                     s=size * 5, alpha=0.7, edgecolors='k', linewidths=0.5)
    ax_final.annotate(f'{size}', gca.positions_final_[i], fontsize=7, ha='center', va='bottom')
ax_final.set_title(f"Partículas finales (tamaño ∝ miembros)", fontweight='bold')

plt.suptitle(f"GCA: {gca.get_params_str()}", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 6. Métricas de calidad y comparación con KMeans

Evaluamos con:
- **Silhouette Score** [-1, 1]: cohesión intra-cluster vs separación inter-cluster (mayor = mejor)
- **Davies-Bouldin Index** [0, +inf): ratio de dispersión/separación (menor = mejor)
- **Calinski-Harabasz** [0, +inf): ratio varianza inter/intra (mayor = mejor)
- **Adjusted Rand Index** [-1, 1]: concordancia con ground truth ajustada por azar (1 = perfecto)

In [ ]:
# --- Celda 7: Métricas y comparación GCA vs KMeans ---

def evaluate_clustering(X, labels, y_true, name=""):
    """Calcula métricas de clustering."""
    valid = labels >= 0
    n_clusters = len(set(labels[valid])) if valid.any() else 0
    
    metrics = {'Algoritmo': name, 'Clusters': n_clusters}
    
    if n_clusters >= 2 and valid.sum() > n_clusters:
        metrics['Silhouette'] = silhouette_score(X[valid], labels[valid])
        metrics['Davies-Bouldin'] = davies_bouldin_score(X[valid], labels[valid])
        metrics['Calinski-Harabasz'] = calinski_harabasz_score(X[valid], labels[valid])
    else:
        metrics['Silhouette'] = np.nan
        metrics['Davies-Bouldin'] = np.nan
        metrics['Calinski-Harabasz'] = np.nan
    
    # ARI contra ground truth (excluyendo outliers del GT)
    gt_clean = y_true >= 0
    both = valid & gt_clean
    metrics['ARI'] = adjusted_rand_score(y_true[both], labels[both]) if both.sum() > 0 else np.nan
    
    return metrics


# --- KMeans ---
t0 = time.time()
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)
km_time = time.time() - t0

# --- Evaluar ambos ---
gca_metrics = evaluate_clustering(X_scaled, gca.labels_, y_true, "GCA")
gca_metrics['Tiempo (s)'] = gca.elapsed_time_
gca_metrics['Iteraciones'] = gca.n_iter_

km_metrics = evaluate_clustering(X_scaled, km_labels, y_true, "KMeans")
km_metrics['Tiempo (s)'] = km_time
km_metrics['Iteraciones'] = kmeans.n_iter_

# --- Tabla comparativa ---
import pandas as pd
df_metrics = pd.DataFrame([gca_metrics, km_metrics]).set_index('Algoritmo')

print("=" * 70)
print("COMPARACIÓN GCA vs KMeans")
print("=" * 70)
display(df_metrics.round(4).T)

# --- Visualización lado a lado ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ground truth
for label in sorted(set(y_true)):
    mask = y_true == label
    lbl = f"C{label}" if label >= 0 else "Outlier"
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1], label=lbl, s=25, alpha=0.7,
                    edgecolors='k', linewidths=0.2)
axes[0].set_title("Ground Truth", fontweight='bold')
axes[0].legend(fontsize=8)

# GCA
for label in sorted(set(gca.labels_)):
    mask = gca.labels_ == label
    axes[1].scatter(X_scaled[mask, 0], X_scaled[mask, 1], label=f"C{label}",
                    s=25, alpha=0.7, edgecolors='k', linewidths=0.2)
sil_gca = gca_metrics['Silhouette']
ari_gca = gca_metrics['ARI']
axes[1].set_title(f"GCA (Sil={sil_gca:.3f}, ARI={ari_gca:.3f})" if not np.isnan(sil_gca) 
                  else f"GCA ({gca.n_clusters_} clusters)", fontweight='bold')
axes[1].legend(fontsize=8)

# KMeans
for label in sorted(set(km_labels)):
    mask = km_labels == label
    axes[2].scatter(X_scaled[mask, 0], X_scaled[mask, 1], label=f"C{label}",
                    s=25, alpha=0.7, edgecolors='k', linewidths=0.2)
axes[2].set_title(f"KMeans (Sil={km_metrics['Silhouette']:.3f}, ARI={km_metrics['ARI']:.3f})",
                  fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle("Comparación: Ground Truth vs GCA vs KMeans", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Robustez: Efecto de outliers

Ejecutamos GCA y KMeans con 0%, 5%, 10% y 20% de outliers para evaluar robustez.

In [ ]:
# --- Celda 8: Robustez frente a outliers ---

outlier_fracs = [0.0, 0.05, 0.10, 0.20]
robustness_results = []

for frac in outlier_fracs:
    X_r, y_r = generate_data(N_POINTS, N_CLUSTERS, frac, NOISE_STD)
    X_r_sc = StandardScaler().fit_transform(X_r)
    
    # Masas por densidad
    nn_r = NearestNeighbors(n_neighbors=11).fit(X_r_sc)
    d_r, _ = nn_r.kneighbors(X_r_sc)
    dens_r = 1.0 / (d_r[:, 1:].mean(axis=1) + 1e-10)
    m_r = (dens_r - dens_r.min()) / (dens_r.max() - dens_r.min()) * 0.9 + 0.1
    
    # GCA
    gca_r = GravitationalClustering(G=0.5, epsilon=1e-2, damping=0.5,
                                     temperature=0.0, max_iter=200, tol=1e-5,
                                     merge_dist=1.0)
    gca_r.fit(X_r_sc, m_r)
    gca_m = evaluate_clustering(X_r_sc, gca_r.labels_, y_r, f"GCA ({frac*100:.0f}%)")
    gca_m['Outlier %'] = frac * 100
    gca_m['Método'] = 'GCA'
    robustness_results.append(gca_m)
    
    # KMeans
    km_r = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
    km_labels_r = km_r.fit_predict(X_r_sc)
    km_m = evaluate_clustering(X_r_sc, km_labels_r, y_r, f"KMeans ({frac*100:.0f}%)")
    km_m['Outlier %'] = frac * 100
    km_m['Método'] = 'KMeans'
    robustness_results.append(km_m)

df_robust = pd.DataFrame(robustness_results)

# --- Gráficos de robustez ---
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
metrics_to_plot = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz', 'ARI']
better_dir = ['higher', 'lower', 'higher', 'higher']

for ax, metric, direction in zip(axes, metrics_to_plot, better_dir):
    for method in ['GCA', 'KMeans']:
        subset = df_robust[df_robust['Método'] == method]
        style = '-o' if method == 'GCA' else '--s'
        ax.plot(subset['Outlier %'], subset[metric], style, label=method, 
                linewidth=2, markersize=8)
    ax.set_xlabel("% Outliers")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} ({'↑' if direction == 'higher' else '↓'} mejor)", fontweight='bold')
    ax.legend()

plt.suptitle("Robustez frente a outliers: GCA vs KMeans", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

display(df_robust[['Algoritmo', 'Outlier %', 'Silhouette', 'Davies-Bouldin', 'ARI', 'Clusters']].round(4))


## 8. Sensibilidad a parámetros G y Temperatura

Barrido paramétrico para entender el efecto de la constante gravitacional y la temperatura.

In [ ]:
# --- Celda 9: Barrido paramétrico G × Temperatura ---

G_values = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
T_values = [0.0, 0.05, 0.1, 0.2, 0.5]

param_results = []

for G_val in G_values:
    for T_val in T_values:
        gca_p = GravitationalClustering(
            G=G_val, epsilon=1e-2, damping=0.5, temperature=T_val,
            max_iter=200, tol=1e-5, merge_dist=1.0,
        )
        gca_p.fit(X_scaled, masses)
        
        valid = gca_p.labels_ >= 0
        n_cl = gca_p.n_clusters_
        
        sil = silhouette_score(X_scaled[valid], gca_p.labels_[valid]) \
              if n_cl >= 2 and valid.sum() > n_cl else np.nan
        
        param_results.append({
            'G': G_val, 'T': T_val,
            'Silhouette': sil,
            'Clusters': n_cl,
            'Iteraciones': gca_p.n_iter_,
        })

df_params = pd.DataFrame(param_results)

# Heatmap de Silhouette
pivot_sil = df_params.pivot(index='T', columns='G', values='Silhouette')
pivot_cl = df_params.pivot(index='T', columns='G', values='Clusters')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im1 = axes[0].imshow(pivot_sil.values, cmap='RdYlGn', aspect='auto', origin='lower')
axes[0].set_xticks(range(len(G_values)))
axes[0].set_xticklabels(G_values)
axes[0].set_yticks(range(len(T_values)))
axes[0].set_yticklabels(T_values)
axes[0].set_xlabel("G (constante gravitacional)")
axes[0].set_ylabel("T (temperatura)")
axes[0].set_title("Silhouette Score", fontweight='bold')
plt.colorbar(im1, ax=axes[0])
for i in range(len(T_values)):
    for j in range(len(G_values)):
        val = pivot_sil.values[i, j]
        if not np.isnan(val):
            axes[0].text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8)

im2 = axes[1].imshow(pivot_cl.values, cmap='viridis', aspect='auto', origin='lower')
axes[1].set_xticks(range(len(G_values)))
axes[1].set_xticklabels(G_values)
axes[1].set_yticks(range(len(T_values)))
axes[1].set_yticklabels(T_values)
axes[1].set_xlabel("G (constante gravitacional)")
axes[1].set_ylabel("T (temperatura)")
axes[1].set_title("Número de clusters", fontweight='bold')
plt.colorbar(im2, ax=axes[1])
for i in range(len(T_values)):
    for j in range(len(G_values)):
        axes[1].text(j, i, f"{int(pivot_cl.values[i, j])}", ha='center', va='center', fontsize=9)

plt.suptitle("Sensibilidad paramétrica del GCA", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Animación del colapso gravitacional

Visualización frame-a-frame de cómo las partículas migran hacia los centros de masa.

In [ ]:
# --- Celda 10: Animación del colapso gravitacional ---

fig_anim, ax_anim = plt.subplots(figsize=(8, 6))

positions_history = gca.history_['positions']
n_frames = len(positions_history)

# Límites fijos basados en rango máximo
all_pos = np.vstack(positions_history)
x_lim = (all_pos[:, 0].min() - 0.5, all_pos[:, 0].max() + 0.5)
y_lim = (all_pos[:, 1].min() - 0.5, all_pos[:, 1].max() + 0.5)

scatter = ax_anim.scatter([], [], c=y_true, cmap='tab10', s=20, alpha=0.7,
                          edgecolors='k', linewidths=0.2)
title = ax_anim.set_title("")
ax_anim.set_xlim(x_lim)
ax_anim.set_ylim(y_lim)
ax_anim.grid(True, alpha=0.3)

def animate(frame):
    pos = positions_history[frame]
    scatter.set_offsets(pos)
    title.set_text(f"GCA — Iteración {frame}/{n_frames - 1}")
    return scatter, title

anim = animation.FuncAnimation(fig_anim, animate, frames=n_frames, 
                                interval=150, blit=True, repeat=True)
plt.close(fig_anim)
HTML(anim.to_jshtml())

## 10. Conclusiones y próximos pasos

**Observaciones clave del algoritmo GCA puro:**

1. **Analogía física completa:** Cada dato es una partícula con masa; la gravedad las atrae y las fusiona naturalmente
2. **Sin dependencias externas:** Los clusters emergen puramente de la fusión gravitacional, sin DBSCAN ni K predefinido
3. **Ventaja sobre KMeans:** No requiere especificar el número de clusters a priori; emerge de la dinámica
4. **Robustez a outliers:** Los outliers tienen baja masa (baja densidad local) → menor influencia gravitacional
5. **Sensibilidad a G:** G alta → colapso rápido en pocos clusters; G baja → fragmentación excesiva
6. **merge_dist es el parámetro más crítico:** Define cuándo dos partículas son "lo suficientemente cercanas" para fusionarse

**Limitaciones del algoritmo base:**
- Complejidad O(n²) por iteración (fuerzas par-a-par)
- Sensible al parámetro `merge_dist`
- No determinista si temperatura > 0

**Próximos pasos (variaciones a explorar):**
- Gravitational Search Algorithm (GSA) — Rashedi et al., 2009
- Kernel-based Gravitational Clustering
- Aproximación Barnes-Hut para escalar a n > 10k
- Masa adaptativa (actualizar masas en cada iteración según fitness)